# Checking the K&K verifier results

This notebook recomputes every number used in the CV from the saved files. It needs no GPU and no model.

**Written with AI assistance.** It only reads saved files and counts; it does not change anything.

**Files:** it reads the files in the `1_current` folder of `kk_verifier_files_2026-10-05` (subfolders `data`, `tests`, `parser_outputs`, `labels_and_reviews`). Upload that folder to Drive and set `DATA_DIR` below if its location differs.

In [2]:
import json, csv, re, collections

from paths import load_paths

DATA_DIR = load_paths().verifier_data_dir   # change this if you put the folder somewhere else

def path(name):
    return f"{DATA_DIR}/{name}"

# On Colab this mounts your Drive; anywhere else it is skipped
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Rollouts kept by the format filter

Expected: 684 rollouts, 503 pass every format rule, 6 more are dropped because their think text uses the worked example's role "sailor", leaving **497 rollouts** from **169 puzzles**.

In [3]:
rows = list(csv.DictReader(open(path("data/rollout_filter_v2_2026-10-05.csv"))))
passed = [r for r in rows if r["kept"] == "yes"]
sailor = [r for r in passed if r["example_role_in_think"] == "yes"]
kept = [r for r in passed if r["example_role_in_think"] != "yes"]





# Why rollouts were removed (one rollout can break more than one rule)
reasons = collections.Counter()
for r in rows:
    if r["kept"] != "yes":
        for x in r["reason_removed"].split(";"):
            if x.strip():
                reasons[x.strip()] += 1
for reason, n in reasons.most_common():
    pass


rollouts: 684
pass every format rule: 503
dropped for using the example role 'sailor': 6
kept: 497 | puzzles covered: 169
 106  tagged answers back to back
  76  step answer is not one sentence plus one clean Name: role tag (a tag holds more than one pair or a statement)
  22  no think before the final answer
  10  final answer is not a clean JSON of the puzzle's characters
   9  step answer is not one sentence plus one clean Name: role tag (a step answer has no tag)
   6  tags nested or text outside blocks
   3  step answer is not one sentence plus one clean Name: role tag (two or more tags in one answer)
   3  step answer is not one sentence plus one clean Name: role tag (a tag's role is not one of the puzzle's two roles)
   2  step answer is not one sentence plus one clean Name: role tag (a tag names someone not in the puzzle)
   1  tags inside a think block
   1  does not start with a think and end with an answer


### 1b. Rerun the format rules on the kept rollouts

These are the same rules the filter used. Every kept rollout should pass, so the last line should say 0.

In [4]:
TAG = re.compile(r"</?(think|answer|sub_ans|final_answer)>")

def blocks(raw):
    """Split the model text into its top-level think and answer blocks, and note any badly nested tags."""
    items, stack, problems = [], [], []
    pos = 0
    for m in TAG.finditer(raw):
        name, closing = m.group(1), m.group(0).startswith("</")
        if not stack and raw[pos:m.start()].strip():
            items.append(("stray", raw[pos:m.start()].strip()))
        if not closing:
            if stack and stack[-1][0] == "think": problems.append(f"<{name}> inside <think>")
            if stack and stack[-1][0] in ("sub_ans", "final_answer"): problems.append(f"<{name}> inside <{stack[-1][0]}>")
            if name in ("think", "answer") and stack: problems.append(f"<{name}> opened inside <{stack[-1][0]}>")
            stack.append((name, m.end(), m.start()))
        else:
            if not stack or stack[-1][0] != name:
                problems.append(f"</{name}> without matching open")
                idx = next((i for i in range(len(stack) - 1, -1, -1) if stack[i][0] == name), None)
                if idx is None:
                    pos = m.end(); continue
                stack = stack[:idx + 1]
            nm, cs, _ = stack.pop()
            if not stack and nm in ("think", "answer"): items.append((nm, raw[cs:m.start()]))
            elif not stack: items.append((nm + "_top", raw[cs:m.start()]))
        pos = m.end()
    tail = raw[pos:]
    if not stack and tail.strip(): items.append(("stray", tail.strip()))
    if stack:
        nm, cs, _ = stack[0]; items.append((nm + "_unclosed", raw[cs:]))
    return items, problems

def answer_parts(c):
    subs = re.findall(r"<sub_ans>(.*?)</sub_ans>", c, re.S)
    fins = re.findall(r"<final_answer>(.*?)</final_answer>", c, re.S)
    rest = re.sub(r"<sub_ans>.*?</sub_ans>|<final_answer>.*?</final_answer>", "", c, flags=re.S)
    return subs, fins, rest.strip()

PAIR = re.compile(r"\s*(\w+)\s*:\s*(\w+)\s*")

def format_problems(raw, names, roles):
    """Return the list of format rules this rollout breaks (empty list = passes)."""
    broken = []; low = [n.lower() for n in names]
    items, problems = blocks(raw)
    if problems or any(k == "stray" for k, _ in items): broken.append("tags nested or text outside blocks")
    think_txt = " ".join(c for k, c in items if k.startswith("think"))
    if re.search(r"</?(answer|sub_ans|final_answer)>", think_txt): broken.append("tags inside a think block")
    seq = [(k, c) for k, c in items if k.startswith(("think", "answer", "final_answer"))]
    kinds = ["T" if k.startswith("think") else "A" for k, _ in seq]
    if not kinds or kinds[0] != "T" or kinds[-1] != "A": broken.append("does not start with a think and end with an answer")
    finals_at = [i for i, (k, c) in enumerate(seq) if k.startswith("final_answer") or "<final_answer>" in c]
    fi = finals_at[-1] if finals_at else None
    if any(kinds[i] == "A" and kinds[i-1] != "T" and i != fi for i in range(1, len(kinds))): broken.append("tagged answers back to back")
    if fi is not None and fi > 0 and kinds[fi - 1] != "T": broken.append("no think before the final answer")
    answers = [c for k, c in seq if not k.startswith("think")]
    if raw.count("<final_answer>") != 1 or raw.count("</final_answer>") != 1: broken.append("final_answer tag not exactly once")
    for c in answers[:-1]:
        subs, fins, rest = answer_parts(c)
        m = PAIR.fullmatch(subs[0]) if len(subs) == 1 else None
        if not (len(subs) == 1 and not fins and rest and m and m.group(1).lower() in low and m.group(2).lower() in roles):
            broken.append("a step answer is not one sentence plus one clean Name: role tag"); break
    if answers:
        last = answers[-1] if not seq[-1][0].startswith("final_answer") else "<final_answer>" + answers[-1] + "</final_answer>"
        fin = re.findall(r"<final_answer>(.*?)</final_answer>", last, re.S)
        try:
            fj = json.loads(fin[-1]) if fin else None
            ok = isinstance(fj, dict) and sorted(x.lower() for x in fj) == sorted(low) and all(isinstance(x, str) and x.lower() in roles for x in fj.values())
        except Exception:
            ok = False
        if "<sub_ans>" in last or not ok: broken.append("final answer is not a clean JSON of the puzzle's characters")
    return broken

steps_file = json.load(open(path("data/steps_think_and_answer_1186_2026-10-05.json")))
puzzle = {p["id"]: p for p in steps_file["puzzles"]}
kept_texts = [json.loads(line) for line in open(path("data/kept_rollouts_v2_497_2026-10-05.jsonl"))]
failing = []
for r in kept_texts:
    p = puzzle[r["task_id"]]
    names, roles = list(p["statements"]), {v.lower() for v in p["roles"].values()}
    if format_problems(r["generated_text"], names, roles):
        failing.append((r["task_id"], r["rollout"]))


kept rollouts rechecked: 497 | failing the format rules now: 0


## 2. Tagged steps

A tagged step is one think-plus-answer pair whose answer carries one `<sub_ans>Name: role</sub_ans>` tag. The final answer is not counted.

Expected: **1,186 tagged steps**; 319 rollouts have 2 tagged steps, 164 have 3 and 14 have 4.

In [5]:
per_rollout = []
for r in kept_texts:
    items, _ = blocks(r["generated_text"])
    answers = [c for k, c in items if k.startswith("answer")]
    n_tags = sum(len(answer_parts(c)[0]) for c in answers[:-1])   # every answer except the final one
    per_rollout.append(n_tags)



tagged steps: 1186
tagged steps per rollout -> number of rollouts: {2: 319, 3: 164, 4: 14}


## 3. The reward paid hedged claims

A hedge is a rollout that tags the same character with both roles in different steps. The `paid` value on each tag is the intermediate reward your reward function gave it.

Expected: **27** rollouts hedge, and in **7** of them the reward paid a tag for the hedged character.

In [6]:
by_rollout = collections.defaultdict(list)
for p in steps_file["puzzles"]:
    for e in p["examples"]:
        name, role = [x.strip().lower() for x in e["tags"][0]["tag"].split(":")]
        by_rollout[(p["id"], e["rollout"])].append((e["step"], name, role, e["tags"][0]["paid"]))

hedged, paid_hedges = 0, []
for key, tags in by_rollout.items():
    roles_of = collections.defaultdict(set)
    for _, name, role, _ in tags:
        roles_of[name].add(role)
    hedged_names = [n for n, rs in roles_of.items() if len(rs) > 1]
    if not hedged_names:
        continue
    hedged += 1
    if any(paid and paid > 0 for _, name, _, paid in tags if name in hedged_names):
        paid_hedges.append((key, sorted(tags)))


for key, tags in paid_hedges:
    pass


rollouts that tag one character with both roles: 27
of these, the reward paid a tag for that character: 7
  ('task_10000', 2) [('step 1', 'elijah: guardian', 'paid 0.0'), ('step 2', 'elijah: deceiver', 'paid 0.1667'), ('step 3', 'ella: deceiver', 'paid 0.1667'), ('step 4', 'scarlett: guardian', 'paid 0.1667')]
  ('task_1013977', 1) [('step 1', 'lucas: saint', 'paid 0.0'), ('step 2', 'lucas: sinner', 'paid 0.25'), ('step 3', 'hannah: sinner', 'paid 0.25')]
  ('task_121903', 2) [('step 1', 'sofia: angel', 'paid 0.1667'), ('step 2', 'andrew: angel', 'paid 0.1667'), ('step 3', 'sebastian: angel', 'paid 0.0'), ('step 4', 'sebastian: devil', 'paid 0.1667')]
  ('task_284382', 3) [('step 1', 'samuel: angel', 'paid 0.1667'), ('step 2', 'layla: angel', 'paid 0.1667'), ('step 3', 'skylar: angel', 'paid 0.0'), ('step 4', 'skylar: devil', 'paid 0.1667')]
  ('task_352033', 3) [('step 1', 'ivy: noble', 'paid 0.0'), ('step 2', 'ivy: rogue', 'paid 0.25'), ('step 3', 'james: rogue', 'paid 0.25')]
  ('ta

## 4. Answer sentences almost never judge a statement

The answer sentence is the step's answer with the tag removed. These are all answer sentences that contain a truth word, followed by the steps whose answer sentence alone engages someone.

Expected: only a handful contain a truth word, and **3 of 1,186** engage a character.

In [7]:
answer_only = json.load(open(path("data/steps_answer_only_1186_2026-10-05.json")))
truth_words = re.compile(r"\b(true|false|lie|lies|lying|liar|truth|truthful|honest|contradict\w*)\b", re.I)
n_steps, engaged_steps = 0, []
for p in answer_only["puzzles"]:
    for e in p["examples"]:
        n_steps += 1
        if truth_words.search(e["text"]):
            pass
        if any(l["engaged"] for l in e["labels"]):
            engaged_steps.append(e["id"])


  task_216665_r1_s1: If Michael is a guardian, his statement is true, so either Grace is a deceiver or Samuel is a guardian.
  task_216665_r1_s2: If Michael is a deceiver, his statement is false, so Grace is a guardian.
  task_386439_r2_s1: Alexander's statement is true, so if Gabriel is a guardian, Sophia must also be a guardian.

answer sentences: 1186 | steps whose answer sentence alone engages someone: 3 ['task_216665_r1_s1', 'task_216665_r1_s2', 'task_386439_r2_s1']


## 5. The parser on the hard test sets

Each set is scored on its own, and boundary labels are kept out of the headline numbers.

Expected: gold **31 of 31**, merged stress **131 of 132**, stress v3 **25 of 25**, so **187 of 188** in total. The one error is `I-02`.

In [8]:
tests = json.load(open(path("tests/engaged_tests_all_2026-10-05.json")))
parser = {r["id"]: r["answer"] for r in json.load(open(path("parser_outputs/out_engaged_tests_all_2026-10-05.json")))}
score = collections.defaultdict(lambda: [0, 0]); wrong, boundary = [], []
for p in tests["puzzles"]:
    for e in p["examples"]:
        for l in e["labels"]:
            said = parser[e["id"]][l["speaker"]]
            if l.get("ratify"):
                boundary.append((p["set"], e["id"], l["speaker"], l["engaged"], said)); continue
            score[p["set"]][0] += said == l["engaged"]; score[p["set"]][1] += 1
            if said != l["engaged"]:
                wrong.append((p["set"], e["id"], l["speaker"], "label", l["engaged"], "parser", said, e["text"]))
for s, (right, total) in score.items():
    pass




gold: 31 of 31 right
stress_merged: 131 of 132 right
stress_v3: 25 of 25 right
total: 187 of 188

wrong labels:
  ('stress_merged', 'I-02', 'Priya', 'label', False, 'parser', True, 'The first statement is false, so Omar is a saint.')

boundary labels (set, block, character, proposed label, parser):
  ('stress_merged', 'H-04', 'Tara', True, True)
  ('stress_v3', 'N-08', 'Noah', True, False)
  ('stress_v3', 'N-04', 'Tara', True, True)


[None, None, None]

## 6. The 200 randomly sampled steps

Expected: **190 of 200** steps engage at least one character, and **248 of 479** character labels are engaged. These labels were proposed by a model and you reviewed 50 of the steps.

In [9]:
sample = json.load(open(path("labels_and_reviews/engagement_labels_sample200_2026-10-05.json")))
steps_with_someone = sum(any(v["engaged"] for v in row["think_only"].values()) for row in sample)
labels = [v["engaged"] for row in sample for v in row["think_only"].values()]



steps: 200 | engage at least one character: 190
character labels: 479 | engaged: 248
